# 05 — Comparison Dashboard

Pure analysis notebook. Loads all experiment results from the shared registry and produces cross-experiment comparisons.

**Run order:** 01 → 02 → 03 → 06 → 04 → this notebook.

In [ ]:
# Pure analysis notebook — reads all experiment results from the shared registry.
# No training happens here. Run notebooks 01, 02, 03, 06, 04 first.
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CONFIG = {
    'registry_path': Path('../../data/results/results_registry.csv'),
}

registry = pd.read_csv(CONFIG['registry_path'])
print(f'Registry: {len(registry)} rows')
print(registry['experiment'].value_counts().to_string())

Registry contents overview. Each row = one model evaluation from one experiment.

## Section 1: Granularity Sweet Spot

In [ ]:
# Line plot: best cv_macro_f1 vs n_classes per game.
# Answers: at what granularity does performance peak before declining?
gran = (registry[registry['experiment']=='granularity_per_dataset']
        .groupby(['train_game','n_classes'])['cv_macro_f1'].max().reset_index())

fig, ax = plt.subplots(figsize=(9, 5))
for game, color in [('WoT','#1565C0'), ('Dota','#C62828')]:
    df = gran[gran['train_game']==game].sort_values('n_classes')
    if df.empty:
        continue
    ax.plot(df['n_classes'], df['cv_macro_f1'], marker='o', linewidth=2,
            color=color, label=game)
    for _, row in df.iterrows():
        ax.annotate(f'{row["cv_macro_f1"]:.3f}',
                    (row['n_classes'], row['cv_macro_f1']),
                    textcoords='offset points', xytext=(0,7), ha='center', fontsize=9)

ax.set_xlabel('Number of Classes', fontsize=12)
ax.set_ylabel('CV Macro F1 (best model per step)', fontsize=12)
ax.set_title('Granularity Sweet Spot — In-Game Performance', fontweight='bold', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
Path('../../data/results').mkdir(parents=True, exist_ok=True)
plt.savefig('../../data/results/dashboard_granularity.png', dpi=150, bbox_inches='tight')
plt.show()

The inflection point is the answer to the core research question. Where F1 peaks and then drops, adding more label granularity hurts generalization — that is the optimal class count.

## Section 2: All Experiments Comparison

In [ ]:
# Bar chart: best test_macro_f1 per experiment per game.
# Compares ML pipeline, error analysis fix, and ensemble results side-by-side.
exp_order = ['ml_pipeline_multiclass', 'error_analysis_fix', 'ensemble']
exp_best  = (registry[registry['experiment'].isin(exp_order)
                       & registry['test_macro_f1'].notna()]
             .groupby(['experiment','train_game'])['test_macro_f1'].max().reset_index())

if exp_best.empty:
    print('No test_macro_f1 results yet — run notebooks 02, 06, 04 first.')
else:
    fig, ax = plt.subplots(figsize=(12, 5))
    x      = np.arange(len(exp_best))
    colors = exp_best['train_game'].map({'WoT':'#1565C0','Dota':'#C62828'}).fillna('#388E3C')
    ax.bar(x, exp_best['test_macro_f1'], color=colors, width=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels(exp_best['experiment'] + '\n(' + exp_best['train_game'] + ')',
                        rotation=15, fontsize=9)
    ax.set_ylabel('Test Macro F1')
    ax.set_title('Best Test Performance per Experiment', fontweight='bold', fontsize=13)
    ax.set_ylim(0, 1)
    for i, v in enumerate(exp_best['test_macro_f1']):
        ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig('../../data/results/dashboard_all_experiments.png', dpi=150, bbox_inches='tight')
    plt.show()

Best test_macro_f1 progression from baseline to fixes to ensemble. Upward trend = each stage of the pipeline adds value.

## Section 3: Anomaly Detection AUROC Summary

In [ ]:
# Table and bar chart of AUROC per anomaly setup and model.
# AUROC > 0.7 = useful signal without labels. Compares IsolationForest vs OneClassSVM.
anomaly = registry[registry['experiment']=='anomaly_detection'][
    ['notes','model','anomaly_auroc']].dropna()

if anomaly.empty:
    print('No anomaly results yet — run notebook 03 first.')
else:
    print(anomaly.sort_values('anomaly_auroc', ascending=False).to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    x = np.arange(len(anomaly))
    color_map = {'IsolationForest': '#1565C0', 'OneClassSVM': '#E65100'}
    colors = [color_map.get(m, '#888888') for m in anomaly['model']]
    ax.bar(x, anomaly['anomaly_auroc'], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels(anomaly['notes'] + '\n' + anomaly['model'], fontsize=8, rotation=15)
    ax.axhline(0.7, color='gray', linestyle='--', linewidth=1, label='AUROC=0.7 threshold')
    ax.set_ylabel('AUROC')
    ax.set_title('Anomaly Detection AUROC per Setup', fontweight='bold')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig('../../data/results/dashboard_anomaly.png', dpi=150, bbox_inches='tight')
    plt.show()

AUROC interpretation: values above the 0.7 threshold indicate the model captures useful toxicity signal without any labels. IsolationForest and OneClassSVM are compared directly. Results below threshold suggest the feature space is insufficient for unsupervised separation.

## Section 4: Full Registry Table

In [ ]:
# Complete registry sorted by test_macro_f1 for reference and paper appendix.
display_cols = ['experiment', 'train_game', 'test_game', 'n_classes', 'model',
                'cv_macro_f1', 'test_macro_f1', 'ood_macro_f1', 'anomaly_auroc', 'notes']
available_cols = [c for c in display_cols if c in registry.columns]
summary = registry[available_cols].sort_values('test_macro_f1', ascending=False, na_position='last')
print(summary.to_string(index=False))
summary.to_csv('../../data/results/full_results_summary.csv', index=False)
print(f'\nSaved to data/results/full_results_summary.csv')

Full results table saved to CSV for paper appendix. All experiments, all models, all metrics in one place.